# Chapter 9: Risk Measurement II in code

Chapter 8 used the risk measures to hedge. Chapter 9 asks a harder question: **what happens to the risk measures themselves when market conditions change?** This notebook recomputes the chapter's numeric examples, then redraws its graphs (Figures 9-1 to 9-24) from a model so you can see where each measure is large, small, positive or negative.
Notes: [[chapters/09-risk-measurement-ii]]

**Set-up (a deliberate simplification).** Black-Scholes is used as a tool, with **interest rate 0 and no dividends**, so the forward price equals the underlying price and "at the money" is clean. The book's graphs are drawn without numbers, so ours show the *shape* of each relationship, not the book's exact values. Unless stated, underlying $S = 100$, volatility 25%, 91 days to expiration.

**Units** (same as Chapter 7): delta and gamma in whole numbers (delta 56 means 0.56); theta per calendar day; vega per 1 percentage point of volatility.

**Sign convention for time.** Every "per day" measure below is the change after **one more day passes** (so time to expiration gets shorter). The book's formulas (Figure 9-26) differentiate with respect to time *to* expiration, which flips the sign; where this matters (color) the notes say so.

Needs `numpy` and `matplotlib` (both in Anaconda). Run it from the `code/` folder: figures are saved to `../images/`.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from math import erf, pi

DAY = 1 / 365

def N(x):
    '''Standard normal cumulative probability (works on arrays).'''
    return 0.5 * (1 + np.vectorize(erf)(np.asarray(x, float) / np.sqrt(2)))

def pdf(x):
    '''Standard normal density (height of the bell curve).'''
    return np.exp(-np.asarray(x, float) ** 2 / 2) / np.sqrt(2 * pi)

def d1_d2(S, K, t, vol):
    d1 = (np.log(S / K) + 0.5 * vol ** 2 * t) / (vol * np.sqrt(t))
    return d1, d1 - vol * np.sqrt(t)

# --- value and the five traditional measures (call; a put has the same gamma, vega, theta when r = 0) ---
def call(S, K, t, vol):
    d1, d2 = d1_d2(S, K, t, vol)
    return S * N(d1) - K * N(d2)

def put(S, K, t, vol):
    return call(S, K, t, vol) - S + K          # put-call parity with r = 0

def delta(S, K, t, vol):                       # call delta, whole numbers
    return 100 * N(d1_d2(S, K, t, vol)[0])

def gamma(S, K, t, vol):                       # change in delta per 1 point of S, whole numbers
    d1, _ = d1_d2(S, K, t, vol)
    return 100 * pdf(d1) / (S * vol * np.sqrt(t))

def vega(S, K, t, vol):                        # value change per 1 vol point
    d1, _ = d1_d2(S, K, t, vol)
    return S * pdf(d1) * np.sqrt(t) / 100

def theta(S, K, t, vol):                       # value change per day (negative for a long option)
    d1, _ = d1_d2(S, K, t, vol)
    return -S * pdf(d1) * vol / (2 * np.sqrt(t)) / 365

# --- the higher-order measures: change in a measure when ONE input moves by a small step ---
def vanna(S, K, t, vol):    return delta(S, K, t, vol + .005) - delta(S, K, t, vol - .005)   # delta per +1 vol point
def charm(S, K, t, vol):    return delta(S, K, t - DAY, vol) - delta(S, K, t, vol)           # delta per day passing
def volga(S, K, t, vol):    return vega(S, K, t, vol + .005) - vega(S, K, t, vol - .005)     # vega per +1 vol point
def vega_decay(S, K, t, vol): return vega(S, K, t - DAY, vol) - vega(S, K, t, vol)           # vega per day passing
def speed(S, K, t, vol):    return (gamma(S + .05, K, t, vol) - gamma(S - .05, K, t, vol)) / .1  # gamma per +1 point of S
def color(S, K, t, vol):    return gamma(S, K, t - DAY, vol) - gamma(S, K, t, vol)           # gamma per day passing
def zomma(S, K, t, vol):    return gamma(S, K, t, vol + .005) - gamma(S, K, t, vol - .005)   # gamma per +1 vol point

def lam(S, K, t, vol):                         # lambda (elasticity) of a call: delta x S / value
    return delta(S, K, t, vol) / 100 * S / call(S, K, t, vol)

def by_delta(fn, t, vol, S=100.0):
    '''Evaluate a measure for calls of every moneyness, and return it against the call delta (2 to 98).'''
    K = np.linspace(40, 260, 4401)
    d = delta(S, K, t, vol)
    keep = (d > 2) & (d < 98)
    return d[keep], fn(S, K[keep], t, vol)

S0, VOL, T = 100.0, 0.25, 91 / 365
print(f"100 call, 91 days, vol 25%: value {float(call(S0, 100, T, VOL)):.3f}  delta {float(delta(S0, 100, T, VOL)):.1f}  "
      f"gamma {float(gamma(S0, 100, T, VOL)):.2f}  theta {float(theta(S0, 100, T, VOL)):.4f}  vega {float(vega(S0, 100, T, VOL)):.4f}")

## The book's own numbers
Four small examples are worked in the text. We redo each one.

### 1. Estimating the theta of an at-the-money option (page 145)
For an at-the-money option, value is proportional to $\sqrt{t}$, so one day later the value is $TV_{t-1} = TV_t\sqrt{(t-1)/t}$ and the theta is $TV_t\left(1-\sqrt{(t-1)/t}\right)$. Book: value 2.50, 30 days left, theta about 0.042; a day later (value 2.458, 29 days left) about 0.043.

In [ ]:
tv = 2.50
for days in (30, 29):
    theta_est = tv * (1 - np.sqrt((days - 1) / days))
    print(f"{days} days left, value {tv:.3f}:  theta ~ {theta_est:.4f}")
    tv -= theta_est

# Check against the model: find the volatility that makes a 30-day at-the-money option worth 2.50 (r = 0)
vol = 2.50 / (0.3989 * 100 * np.sqrt(30 / 365))       # at the money with r = 0: value = 0.3989 x S x vol x sqrt(t)
v30, v29, v28 = (call(100, 100, d / 365, vol) for d in (30, 29, 28))
print(f"model volatility {vol:.1%}: value at 30 days {v30:.3f}; true decay over the next day {v30 - v29:.4f}; the day after {v29 - v28:.4f}")

### 2. Lambda (page 154)
A call worth 4.00 with a delta of 20, underlying at 100: $\Lambda = 0.20 \times 100/4.00 = 5$. The underlying rises 1% (to 101); the option gains about $0.20$, which is 5% of 4.00.

In [ ]:
S, tv, dlt = 100, 4.00, 0.20
print("lambda =", dlt * S / tv)
print(f"underlying +1% -> option +{100 * dlt * (0.01 * S) / tv:.1f}%")

### 3. An implied delta that moves with implied volatility (page 137)
40 calls, implied volatility 32%, implied delta 25 each, hedged by selling 10 underlying contracts: $40 \times 25 - 10 \times 100 = 0$. The book then raises implied volatility to 36% and assumes the delta becomes 30, giving $40 \times 30 - 10 \times 100 = +200$. The 30 is a round number; here is what a model gives for a 91-day call with delta 25 at 32%.

In [ ]:
K = np.linspace(100, 200, 20001)
K25 = K[np.argmin(abs(delta(100, K, T, 0.32) - 25))]
d36 = float(delta(100, K25, T, 0.36))
print(f"strike {K25:.1f}: delta {float(delta(100, K25, T, 0.32)):.1f} at 32%, {d36:.1f} at 36%")
print(f"hedged with 10 short contracts: position delta = 40 x {d36:.1f} - 1000 = {40 * d36 - 1000:+.0f}   (book, using delta 30: +200)")
print(f"vanna at 34% = {float(vanna(100, K25, T, 0.34)):.2f} delta points per vol point, so 4 points of volatility -> about +{4 * float(vanna(100, K25, T, 0.34)):.1f}")

### 4. At-the-money gamma depends on time and volatility (page 151)
A 100 call with the underlying at 97.50. The book moves the underlying to 102.50 (5 points) and compares two cases: delta 25 to 75 (gamma $=50/5=10$) and delta 45 to 55 (gamma $=10/5=2$). We search for a time and a volatility that reproduce each.

In [ ]:
for label, vol, days in [("short time / low volatility", 0.20, 13), ("long time / high volatility", 0.40, 91)]:
    a, b = float(delta(97.5, 100, days / 365, vol)), float(delta(102.5, 100, days / 365, vol))
    print(f"{label:30s} vol {vol:.0%}, {days:>2} days: delta {a:.1f} -> {b:.1f}, gamma ~ {(b - a) / 5:.2f}; gamma at 100 = {float(gamma(100, 100, days / 365, vol)):.2f}")
print("(book: 25 -> 75, gamma 10; and 45 -> 55, gamma 2. The long-time case gives 49 -> 59 here because the lognormal tilt keeps the delta above 50, but the gamma of 2 matches.)")

## Delta (Figures 9-1 to 9-7)
**Delta against volatility and time.** As volatility rises or time grows, calls move toward delta 50: an out-of-the-money call's delta rises, an in-the-money call's delta falls, and an at-the-money call stays near 50 (slightly above, because of the lognormal tilt).

In [ ]:
vols = np.linspace(0.05, 0.80, 100)
days = np.linspace(365, 1, 365)
fig, (a1, a2) = plt.subplots(1, 2, figsize=(11, 4.4))
for K, lab in [(90, "in the money (90 call)"), (100, "at the money (100 call)"), (110, "out of the money (110 call)")]:
    a1.plot(vols * 100, delta(100, K, T, vols), label=lab)
    a2.plot(days, delta(100, K, days / 365, 0.25), label=lab)
a1.set_xlabel("volatility (%)"); a1.set_ylabel("call delta"); a1.set_title("Call delta as volatility changes (Figure 9-1)")
a2.set_xlabel("days to expiration (time passes to the right)"); a2.invert_xaxis(); a2.set_title("Call delta as time passes (Figure 9-2)")
for a, where in ((a1, "upper right"), (a2, "upper left")):
    a.axhline(50, color="grey", ls=":", lw=1); a.set_ylim(0, 100); a.legend(fontsize=8, loc=where)
plt.tight_layout(); plt.savefig("../images/ch09-delta-vs-volatility-and-time.png", dpi=130); plt.show()

print("call delta by volatility (91 days):  vol    90 call  100 call  110 call")
for v in (0.10, 0.15, 0.25, 0.40, 0.60):
    print(f"                                    {v:.0%}   {float(delta(100, 90, T, v)):6.1f}  {float(delta(100, 100, T, v)):7.1f}  {float(delta(100, 110, T, v)):7.1f}")
print("call delta by days left (vol 25%):  days   90 call  100 call  110 call")
for d in (180, 90, 30, 14, 7, 3):
    print(f"                                    {d:>3}   {float(delta(100, 90, d / 365, .25)):6.1f}  {float(delta(100, 100, d / 365, .25)):7.1f}  {float(delta(100, 110, d / 365, .25)):7.1f}")

**Vanna and charm.** Vanna is the change in delta per 1 volatility point; charm is the change in delta per day that passes. Both are zero near delta 50 and largest for calls with deltas of about 15 to 20 and 80 to 85 (puts mirror this: $-15$ to $-20$ and $-80$ to $-85$). We plot calls against their delta, so the picture is the same for every underlying price and strike.

In [ ]:
fig, (a1, a2) = plt.subplots(1, 2, figsize=(11, 4.4))
for vol in (0.15, 0.25, 0.40):
    x, y = by_delta(vanna, T, vol); a1.plot(x, y, label=f"volatility {vol:.0%}")
for days_, lab in [(30, "1 month left"), (91, "3 months left"), (365, "1 year left")]:
    x, y = by_delta(charm, days_ / 365, 0.25); a2.plot(x, y, label=lab)
a1.set_title("Vanna: delta change per +1 vol point (Figure 9-6)"); a2.set_title("Charm: delta change per day passing (Figure 9-7)")
for a in (a1, a2):
    a.axhline(0, color="black", lw=0.8); a.set_xlabel("call delta"); a.legend(fontsize=8)
plt.tight_layout(); plt.savefig("../images/ch09-vanna-and-charm.png", dpi=130); plt.show()

# "Both interpretations are mathematically identical": vanna is also the change in vega per 1 point of underlying
K = 110
dv = (vega(100.5, K, T, 0.25) - vega(99.5, K, T, 0.25))        # vega change per +1 point of the underlying
print(f"change in vega per +1 underlying point = {float(dv):.4f}; vanna / 100 = {float(vanna(100, K, T, .25)) / 100:.4f}  (the /100 is only units: delta in whole numbers)")

## Theta (Figures 9-8 to 9-11)
Theta is largest (in size) at the money. At the money it grows without limit as expiration nears; in or out of the money it fades to zero. With no interest, at-the-money theta is exactly proportional to volatility.

In [ ]:
unders = np.linspace(60, 140, 161)
daysx = np.linspace(120, 1, 240)
fig, axs = plt.subplots(1, 3, figsize=(14, 4.4))
for K, lab in [(90, "strike 90"), (100, "strike 100"), (110, "strike 110")]:
    axs[0].plot(unders, -theta(unders, K, T, 0.25), label=lab)
for K, lab in [(90, "in the money (90 call)"), (100, "at the money (100 call)"), (110, "out of the money (110 call)")]:
    axs[1].plot(daysx, -theta(100, K, daysx / 365, 0.25), label=lab)
for K, lab in [(90, "strike 90"), (100, "at the money (100)"), (110, "strike 110")]:
    axs[2].plot(vols * 100, -theta(100, K, T, vols), label=lab)
axs[0].set_title("|Theta| as the underlying moves (Figure 9-8)"); axs[0].set_xlabel("underlying price")
axs[1].set_title("|Theta| as time passes (Figure 9-10)"); axs[1].set_xlabel("days to expiration"); axs[1].invert_xaxis()
axs[2].set_title("|Theta| as volatility changes (Figure 9-11)"); axs[2].set_xlabel("volatility (%)")
for a in axs:
    a.set_ylabel("theta per day (absolute value)"); a.legend(fontsize=8)
plt.tight_layout(); plt.savefig("../images/ch09-theta-profiles.png", dpi=130); plt.show()

print("theta per day, 100 underlying (negative = value lost):  days left    90 call    100 call   110 call")
for d in (180, 90, 30, 15, 7, 3, 1):
    print(f"                                                  {d:>4}   {float(theta(100, 90, d / 365, .25)):9.4f}  {float(theta(100, 100, d / 365, .25)):9.4f}  {float(theta(100, 110, d / 365, .25)):9.4f}")
print("at-the-money theta against volatility (91 days):", ", ".join(f"{v:.0%}: {float(theta(100, 100, T, v)):.4f}" for v in (0.10, 0.20, 0.40)))
print("theta of 110 vs 90 strike (91 days):", ", ".join(f"vol {v:.0%}: {float(theta(100, 110, T, v)):.4f} vs {float(theta(100, 90, T, v)):.4f}" for v in (0.20, 0.30, 0.40)))

## Vega (Figures 9-12 to 9-17)
Vega is largest at the money and is bigger for higher exercise prices. In-the-money and out-of-the-money vega rises with volatility, because those options start to act like at-the-money options (**volga**), and vega falls as time passes (**vega decay**).

In [ ]:
fig, axs = plt.subplots(1, 3, figsize=(14, 4.4))
for K, lab in [(90, "strike 90"), (100, "strike 100"), (110, "strike 110")]:
    axs[0].plot(unders, vega(unders, K, T, 0.25), label=lab)
for K, lab in [(90, "in the money (90 call)"), (100, "at the money (100 call)"), (110, "out of the money (110 call)")]:
    axs[1].plot(vols * 100, vega(100, K, T, vols), label=lab)
    axs[2].plot(daysx, vega(100, K, daysx / 365, 0.25), label=lab)
axs[0].set_title("Vega as the underlying moves (Figure 9-12)"); axs[0].set_xlabel("underlying price")
axs[1].set_title("Vega as volatility changes (Figure 9-14)"); axs[1].set_xlabel("volatility (%)")
axs[2].set_title("Vega as time passes (Figure 9-16)"); axs[2].set_xlabel("days to expiration"); axs[2].invert_xaxis()
for a in axs:
    a.set_ylabel("vega"); a.legend(fontsize=8)
plt.tight_layout(); plt.savefig("../images/ch09-vega-profiles.png", dpi=130); plt.show()

print("vega by volatility (91 days):  vol    90 call  100 call  110 call")
for v in (0.10, 0.15, 0.25, 0.40, 0.60):
    print(f"                              {v:.0%}   {float(vega(100, 90, T, v)):7.3f}  {float(vega(100, 100, T, v)):8.3f}  {float(vega(100, 110, T, v)):8.3f}")

In [ ]:
fig, (a1, a2) = plt.subplots(1, 2, figsize=(11, 4.4))
for days_, lab in [(30, "1 month left"), (91, "3 months left"), (365, "1 year left")]:
    x, y = by_delta(volga, days_ / 365, 0.25); a1.plot(x, y, label=lab)
    x, y = by_delta(vega_decay, days_ / 365, 0.25); a2.plot(x, y, label=lab)
a1.set_title("Volga: vega change per +1 vol point (Figure 9-15)"); a2.set_title("Vega decay: vega change per day passing (Figure 9-17)")
for a in (a1, a2):
    a.axhline(0, color="black", lw=0.8); a.set_xlabel("call delta"); a.legend(fontsize=8)
plt.tight_layout(); plt.savefig("../images/ch09-volga-and-vega-decay.png", dpi=130); plt.show()

## Gamma (Figures 9-18 to 9-22)
Gamma is largest at the money. At the money, gamma **rises** as time passes or volatility falls (the peak gets taller and narrower); away from the money it behaves the other way. Unlike theta and vega, at-the-money gamma gets **smaller** at higher exercise prices.

In [ ]:
fig, axs = plt.subplots(1, 3, figsize=(14, 4.4))
for d_, lab in [(14, "14 days"), (91, "91 days"), (365, "365 days")]:
    axs[0].plot(unders, gamma(unders, 100, d_ / 365, 0.25), label=lab)
for v in (0.15, 0.25, 0.40):
    axs[1].plot(unders, gamma(unders, 100, T, v), label=f"volatility {v:.0%}")
axs[0].set_title("Gamma of a 100 call: time (Figure 9-20)"); axs[1].set_title("Gamma of a 100 call: volatility (Figure 9-20)")
for a in axs[:2]:
    a.set_xlabel("underlying price"); a.set_ylabel("gamma"); a.legend(fontsize=8)

# at-the-money options with different exercise prices (underlying = strike), relative to the strike-100 option
Ks = np.linspace(25, 200, 176)
axs[2].plot(Ks, vega(Ks, Ks, T, .25) / vega(100, 100, T, .25), label="vega (proportional to strike)")
axs[2].plot(Ks, theta(Ks, Ks, T, .25) / theta(100, 100, T, .25), ls="--", label="theta (proportional to strike)")
axs[2].plot(Ks, gamma(Ks, Ks, T, .25) / gamma(100, 100, T, .25), label="gamma (inversely proportional)")
axs[2].set_title("At-the-money options: ratio to the 100 option"); axs[2].set_xlabel("exercise price (underlying = exercise price)")
axs[2].set_ylim(0, 4); axs[2].legend(fontsize=8)
plt.tight_layout(); plt.savefig("../images/ch09-gamma-profiles.png", dpi=130); plt.show()

print("strike scaling, at the money, 91 days, vol 25%:   strike   value   theta    vega    gamma")
for X in (50, 100, 1000):
    print(f"                                                 {X:>5}  {float(call(X, X, T, .25)):7.3f} {float(theta(X, X, T, .25)):8.4f} {float(vega(X, X, T, .25)):7.4f} {float(gamma(X, X, T, .25)):8.3f}")
print()
print("gamma by days left (underlying 100, vol 25%):  days  90 call  100 call  110 call")
for d in (365, 180, 91, 30, 14, 7, 3, 1):
    print(f"                                              {d:>4}  {float(gamma(100, 90, d / 365, .25)):7.2f}  {float(gamma(100, 100, d / 365, .25)):8.2f}  {float(gamma(100, 110, d / 365, .25)):8.2f}")
print("gamma by volatility (91 days):  vol    90 call  100 call  110 call")
for v in (0.10, 0.15, 0.25, 0.40, 0.60):
    print(f"                               {v:.0%}   {float(gamma(100, 90, T, v)):7.2f}  {float(gamma(100, 100, T, v)):8.2f}  {float(gamma(100, 110, T, v)):8.2f}")

In [ ]:
fig, axs = plt.subplots(1, 3, figsize=(14, 4.4))
for days_, lab in [(30, "1 month left"), (91, "3 months left"), (365, "1 year left")]:
    for a, fn in zip(axs, (speed, color, zomma)):
        x, y = by_delta(fn, days_ / 365, 0.25); a.plot(x, y, label=lab)
axs[0].set_title("Speed: gamma per +1 point (9-19)")
axs[1].set_title("Color: gamma per day passing (9-21)")
axs[2].set_title("Zomma: gamma per +1 vol point (9-22)")
for a in axs:
    a.axhline(0, color="black", lw=0.8); a.set_xlabel("call delta"); a.legend(fontsize=8)
plt.tight_layout(); plt.savefig("../images/ch09-speed-color-zomma.png", dpi=130); plt.show()

**The riskiest options.** An at-the-money option close to expiration in a low-volatility market has an enormous gamma: its delta swings from near 0 to near 100 over a one- or two-point move.

In [ ]:
print("delta of an at-the-money 100 call as the underlying moves (strike 100)")
print("underlying:                         98     99    100    101    102      gamma at 100")
for d_, v in [(91, .25), (30, .25), (3, .15), (1, .15)]:
    row = "  ".join(f"{float(delta(s, 100, d_ / 365, v)):5.1f}" for s in (98, 99, 100, 101, 102))
    print(f"{d_:>3} days, vol {v:.0%}:                {row}        {float(gamma(100, 100, d_ / 365, v)):6.2f}")

## Lambda (Figures 9-23 and 9-24)
Lambda $=\Delta \times S/\text{value}$ is the option's percentage change per 1% change in the underlying. It is largest for cheap out-of-the-money options, and rises as volatility falls or as expiration nears.

In [ ]:
Sx = np.linspace(75, 125, 101)
fig, (a1, a2) = plt.subplots(1, 2, figsize=(11, 4.4))
for v in (0.15, 0.25, 0.40):
    a1.plot(Sx, lam(Sx, 100, T, v), label=f"volatility {v:.0%}")
    put_lam = -(100 - delta(Sx, 100, T, v)) / 100 * Sx / put(Sx, 100, T, v)
    a2.plot(Sx, put_lam, label=f"volatility {v:.0%}")
a1.set_title("Lambda of a 100 call (Figure 9-23)"); a2.set_title("Lambda of a 100 put (Figure 9-24)")
for a in (a1, a2):
    a.set_xlabel("underlying price"); a.set_ylabel("lambda"); a.legend(fontsize=8)
a1.set_ylim(0, 60); a2.set_ylim(-60, 0)
plt.tight_layout(); plt.savefig("../images/ch09-lambda.png", dpi=130); plt.show()

print("call lambda (underlying 100):  strike  days  vol   value   delta  lambda")
for K, d_, v in [(90, 91, .25), (100, 91, .25), (110, 91, .25), (110, 91, .15), (110, 91, .40), (110, 30, .25), (110, 7, .25)]:
    tt = d_ / 365
    print(f"                              {K:>5}  {d_:>4}  {v:.0%}  {float(call(100, K, tt, v)):6.3f}  {float(delta(100, K, tt, v)):5.1f}  {float(lam(100, K, tt, v)):6.1f}")

## Where does each measure peak?
The book's summary (Figures 9-25 and 9-26) lists, for each measure, the conditions under which it is maximized. We find the call delta at which each higher-order measure is largest in size (91 days, volatility 25%) and compare.

In [ ]:
rows = [("vanna", vanna, "15-20 and 80-85"), ("charm", charm, "15-20 and 80-85"), ("volga", volga, "10 and 90"),
        ("vega decay", vega_decay, "20 and 80"), ("speed", speed, "15-20 and 80-85"),
        ("color", color, "at the money, and near 5 and 95"), ("zomma", zomma, "at the money, and near 5 and 95")]
print(f"{'measure':11s} {'largest in size at call delta':>32s}   {'zero at call delta':>20s}   book")
for name, fn, book in rows:
    x, y = by_delta(fn, T, VOL)              # x runs from low delta to high delta
    lo, hi = x < 50, x > 50
    peaks = sorted({round(float(x[lo][np.argmax(abs(y[lo]))])), round(float(x[hi][np.argmax(abs(y[hi]))]))})
    flips = np.where(np.diff(np.sign(y)) != 0)[0]
    zeros = ", ".join(f"{x[k]:.0f}" for k in flips) or "never"
    if name in ("color", "zomma"):
        peaks = [round(float(x[np.argmax(abs(y))]))]          # the single biggest value, at the money
    print(f"{name:11s} {' and '.join(map(str, peaks)):>32s}   {zeros:>20s}   {book}")
print("(color and zomma: the biggest value is at the money; the second peak, of opposite sign, is out near delta 5 and 95.)")

## Summary
- Every sensitivity changes with price, volatility and time; the names (vanna, charm, volga, vega decay, speed, color, zomma) are just labels for those changes.
- Time and volatility usually push in the same direction: more time acts like more volatility.
- Gamma, theta and vega are all largest at the money; at-the-money theta and vega scale with the exercise price, gamma scales inversely.
- The riskiest options are at-the-money, close to expiration, in a quiet market: huge gamma, fast-moving delta.